In [1]:
%pip install openpyxl 
%pip install pandas 
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import ipywidgets as widgets
import io
from pathlib import Path
from IPython.display import display

In [11]:
def baca_file(uploader, label):
    if not uploader.value:
        raise ValueError(f"File {label} belum diupload!")
    item = uploader.value[0]
    nama = item['name']
    buf  = io.BytesIO(item['content'])
    ext  = Path(nama).suffix.lower()

    if ext == ".xlsx":
        df = pd.read_excel(buf, engine="openpyxl")
    elif ext == ".xls":
        df = pd.read_excel(buf, engine="xlrd")
    elif ext in (".csv", ".tsv"):
        sep = "\t" if ext == ".tsv" else ","
        for enc in ("utf-8", "utf-8-sig", "latin-1"):
            try:
                buf.seek(0)
                df = pd.read_csv(buf, sep=sep, encoding=enc)
                break
            except UnicodeDecodeError:
                continue
        else:
            raise ValueError(f"Tidak dapat membaca: {nama}")
    else:
        raise ValueError(f"Format tidak didukung: '{ext}'")

    print(f"✅ '{nama}' — {len(df):,} baris, {len(df.columns)} kolom")
    return df


def validasi_kolom(df, nama):
    kurang = {"sent_id", "misc", "tipe_afiks"} - set(df.columns)
    if kurang:
        raise ValueError(f"File '{nama}' kekurangan kolom: {kurang}\nKolom ada: {list(df.columns)}")

In [12]:
uploader_patokan = widgets.FileUpload(
    accept='.xlsx,.xls,.csv,.tsv', multiple=False, description='📂 Patokan')
uploader_cek = widgets.FileUpload(
    accept='.xlsx,.xls,.csv,.tsv', multiple=False, description='📂 Dicek')

display(widgets.VBox([
    widgets.Label("File patokan (ground truth):"),
    uploader_patokan,
    widgets.Label("File yang dicek:"),
    uploader_cek,
]))

In [24]:
print("Membaca file patokan...")
df_patokan = baca_file(uploader_patokan, "patokan")
validasi_kolom(df_patokan, "patokan")

print("\nMembaca file yang dicek...")
df_cek = baca_file(uploader_cek, "dicek")
validasi_kolom(df_cek, "dicek")

# Normalisasi semua kolom kunci
for df in [df_patokan, df_cek]:
    df["sent_id"]    = df["sent_id"].astype(str).str.strip()
    df["misc"]       = df["misc"].astype(str).str.strip()
    df["tipe_afiks"] = df["tipe_afiks"].astype(str).str.strip()

print("\n── Preview patokan ──")
display(df_patokan[["sent_id", "token", "misc", "tipe_afiks"]].head())
print("── Preview file dicek ──")
display(df_cek[["sent_id", "token", "misc", "tipe_afiks"]].head())

Membaca file patokan...
✅ 'pemeriksaan_morfologiAfiksasi_sample_test.xlsx' — 385 baris, 12 kolom

Membaca file yang dicek...
✅ 'morfologiAfiksasi_sample_test.xlsx' — 385 baris, 12 kolom

── Preview patokan ──


,sent_id,token,misc,tipe_afiks
0,test-s465,tegas,{'MorphInd': '^tegas<a>_ASP$'},Tidak Berimbuhan
1,test-s304,dari,{'MorphInd': '^dari<r>_R--$'},Tidak Berimbuhan
2,test-s62,muda,{'MorphInd': '^muda<a>_ASP$'},Tidak Berimbuhan
3,test-s484,panen,{'MorphInd': '^panen<n>_NSD$'},Tidak Berimbuhan
4,test-s6,merupakan,{'MorphInd': '^merupakan<o>_O--$'},Tidak Berimbuhan


── Preview file dicek ──


,sent_id,token,misc,tipe_afiks
0,test-s465,tegas,{'MorphInd': '^tegas<a>_ASP$'},Tidak Berimbuhan
1,test-s304,dari,{'MorphInd': '^dari<r>_R--$'},Tidak Berimbuhan
2,test-s62,muda,{'MorphInd': '^muda<a>_ASP$'},Tidak Berimbuhan
3,test-s484,panen,{'MorphInd': '^panen<n>_NSD$'},Tidak Berimbuhan
4,test-s6,merupakan,{'MorphInd': '^merupakan<o>_O--$'},Tidak Berimbuhan


In [25]:
hasil_beda    = []
tidak_ketemu  = []

for _, baris in df_patokan.iterrows():
    sid       = baris["sent_id"]
    misc      = baris["misc"]
    tipe_pat  = baris["tipe_afiks"]
    token     = str(baris.get("token", "")).strip()

    # Cari baris di file cek: sent_id sama LALU misc sama
    baris_cek = df_cek[
        (df_cek["sent_id"] == sid) &
        (df_cek["misc"]    == misc)
    ]

    if baris_cek.empty:
        tidak_ketemu.append({
            "sent_id" : sid,
            "token"   : token,
            "misc"    : misc,
            "tipe_patokan": tipe_pat,
        })
    else:
        tipe_cek = baris_cek.iloc[0]["tipe_afiks"]
        if tipe_pat != tipe_cek:
            hasil_beda.append({
                "sent_id"     : sid,
                "token"       : token,
                "misc"        : misc,
                "tipe_patokan": tipe_pat,
                "tipe_cek"    : tipe_cek,
            })

total   = len(df_patokan)
beda    = len(hasil_beda)
tdk_ada = len(tidak_ketemu)
cocok   = total - beda - tdk_ada
akurasi = cocok / total * 100 if total > 0 else 0

print("=" * 50)
print(f"  Baris patokan   : {total:>6,}")
print(f"  Cocok           : {cocok:>6,}")
print(f"  Berbeda         : {beda:>6,}")
print(f"  Tidak ditemukan : {tdk_ada:>6,}")
print(f"\n  🎯 Akurasi      : {akurasi:>6.2f}%")
print("=" * 50)

  Baris patokan   :    385
  Cocok           :    384
  Berbeda         :      1
  Tidak ditemukan :      0

  🎯 Akurasi      :  99.74%


In [26]:
if beda == 0 and tdk_ada == 0:
    print("✅ Tidak ada perbedaan! Semua anotasi sesuai patokan.")

if beda > 0:
    print(f"⚠️  {beda} baris BERBEDA:")
    print(f"{'='*70}")
    print(f"{'No':<5} {'sent_id':<18} {'token':<18} {'patokan':<22} {'cek'}")
    print(f"{'-'*5} {'-'*18} {'-'*18} {'-'*22} {'-'*18}")
    for i, row in enumerate(hasil_beda):
        print(f"{i+1:<5} {row['sent_id']:<18} {row['token']:<18} "
              f"{row['tipe_patokan']:<22} {row['tipe_cek']}")
    print(f"{'='*70}")

if tdk_ada > 0:
    print(f"\n📌 {tdk_ada} baris TIDAK DITEMUKAN di file cek:")
    print(f"{'='*70}")
    print(f"{'No':<5} {'sent_id':<18} {'token':<18} {'misc'}")
    print(f"{'-'*5} {'-'*18} {'-'*18} {'-'*40}")
    for i, row in enumerate(tidak_ketemu):
        print(f"{i+1:<5} {row['sent_id']:<18} {row['token']:<18} {row['misc']}")
    print(f"{'='*70}")

⚠️  1 baris BERBEDA:
No    sent_id            token              patokan                cek
----- ------------------ ------------------ ---------------------- ------------------
1     test-s113          tersebut           Prefiks                Tidak Berimbuhan


# Converter

In [2]:
import pandas as pd
from pathlib import Path

# =========================
# 1. PATH INPUT EXCEL
# =========================
train_path = Path(r"D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\Anotasi\Hasil\Negasi\Annotator 2_Nanda Tamara\negasiScope_sample_train.xlsx")
val_path   = Path(r"D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\Anotasi\Hasil\Negasi\Annotator 2_Nanda Tamara\negasiScope_sample_dev.xlsx")
test_path  = Path(r"D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\Anotasi\Hasil\Negasi\Annotator 2_Nanda Tamara\negasiScope_sample_test.xlsx")

# =========================
# 2. PATH OUTPUT FOLDER
# =========================
output_dir = Path(r"D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\Anotasi\Hasil\Negasi\Annotator 2_Nanda Tamara\csv")
output_dir.mkdir(parents=True, exist_ok=True)

# =========================
# 3. DAFTAR FILE INPUT
# =========================
input_files = [train_path, val_path, test_path]

# =========================
# 4. KONVERSI EXCEL KE CSV
# =========================
for input_path in input_files:
    df = pd.read_excel(input_path)

    # Nama CSV mengikuti nama file Excel
    output_path = output_dir / f"{input_path.stem}.csv"

    df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print(f"Berhasil mengonversi: {input_path.name} -> {output_path}")

Berhasil mengonversi: negasiScope_sample_train.xlsx -> D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\Anotasi\Hasil\Negasi\Annotator 2_Nanda Tamara\csv\negasiScope_sample_train.csv
Berhasil mengonversi: negasiScope_sample_dev.xlsx -> D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\Anotasi\Hasil\Negasi\Annotator 2_Nanda Tamara\csv\negasiScope_sample_dev.csv
Berhasil mengonversi: negasiScope_sample_test.xlsx -> D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\Anotasi\Hasil\Negasi\Annotator 2_Nanda Tamara\csv\negasiScope_sample_test.csv


# Negasi

In [7]:
upload_patokan = widgets.FileUpload(
    accept='.xlsx,.xls,.csv,.tsv',
    multiple=False,
    description='Upload Patokan'
)

upload_cek = widgets.FileUpload(
    accept='.xlsx,.xls,.csv,.tsv',
    multiple=False,
    description='Upload Cek'
)

display(widgets.HTML("<b>Upload file patokan:</b>"))
display(upload_patokan)

display(widgets.HTML("<b>Upload file yang akan dicek:</b>"))
display(upload_cek)

HTML(value='<b>Upload file patokan:</b>')

FileUpload(value=(), accept='.xlsx,.xls,.csv,.tsv', description='Upload Patokan')

HTML(value='<b>Upload file yang akan dicek:</b>')

FileUpload(value=(), accept='.xlsx,.xls,.csv,.tsv', description='Upload Cek')

In [15]:
def get_uploaded_file(upload_widget):
    if not upload_widget.value:
        raise ValueError("Belum ada file yang diupload.")

    uploaded = upload_widget.value

    # Untuk ipywidgets versi baru
    if isinstance(uploaded, tuple):
        file_info = uploaded[0]
        filename = file_info["name"]
        content = file_info["content"]

    # Untuk ipywidgets versi lama
    elif isinstance(uploaded, dict):
        filename = list(uploaded.keys())[0]
        content = uploaded[filename]["content"]

    else:
        raise ValueError("Format upload tidak dikenali.")

    return filename, content


def read_uploaded_file(upload_widget):
    filename, content = get_uploaded_file(upload_widget)
    ext = filename.lower().split(".")[-1]

    if ext in ["xlsx", "xls"]:
        df = pd.read_excel(io.BytesIO(content))
    elif ext == "csv":
        df = pd.read_csv(io.BytesIO(content))
    elif ext == "tsv":
        df = pd.read_csv(io.BytesIO(content), sep="\t")
    else:
        raise ValueError(f"Format file tidak didukung: {ext}")

    return filename, df

In [16]:
def cek_negasi_scope(df_patokan, df_cek):
    required_columns = ["sent_id", "token_id", "negasi_scope"]

    for col in required_columns:
        if col not in df_patokan.columns:
            raise ValueError(f"Kolom '{col}' tidak ditemukan di file patokan.")
        if col not in df_cek.columns:
            raise ValueError(f"Kolom '{col}' tidak ditemukan di file cek.")

    df_patokan = df_patokan.copy()
    df_cek = df_cek.copy()

    # Normalisasi kolom kunci
    for df in [df_patokan, df_cek]:
        df["sent_id"] = df["sent_id"].astype(str).str.strip()
        df["token_id"] = df["token_id"].astype(str).str.strip()
        df["negasi_scope"] = df["negasi_scope"].astype(str).str.strip()

    # Cek duplikat kunci
    duplicate_patokan = df_patokan.duplicated(subset=["sent_id", "token_id"]).sum()
    duplicate_cek = df_cek.duplicated(subset=["sent_id", "token_id"]).sum()

    if duplicate_patokan > 0:
        print(f"⚠️ Ada {duplicate_patokan} duplikat sent_id + token_id pada file patokan.")

    if duplicate_cek > 0:
        print(f"⚠️ Ada {duplicate_cek} duplikat sent_id + token_id pada file cek.")

    # Merge berdasarkan sent_id + token_id
    merged = df_patokan.merge(
        df_cek,
        on=["sent_id", "token_id"],
        how="left",
        suffixes=("_patokan", "_cek"),
        indicator=True
    )

    # Kondisi tidak ditemukan
    tidak_ditemukan = merged[merged["_merge"] == "left_only"].copy()

    # Kondisi ditemukan
    ditemukan = merged[merged["_merge"] == "both"].copy()

    # Cocok dan berbeda
    cocok = ditemukan[
        ditemukan["negasi_scope_patokan"] == ditemukan["negasi_scope_cek"]
    ].copy()

    berbeda = ditemukan[
        ditemukan["negasi_scope_patokan"] != ditemukan["negasi_scope_cek"]
    ].copy()

    total_patokan = len(df_patokan)
    jumlah_cocok = len(cocok)
    jumlah_berbeda = len(berbeda)
    jumlah_tidak_ditemukan = len(tidak_ditemukan)

    akurasi = (jumlah_cocok / total_patokan) * 100 if total_patokan > 0 else 0

    print("=" * 50)
    print("HASIL PENGECEKAN NEGATION SCOPE DETECTION")
    print("=" * 50)
    print(f"Baris patokan   : {total_patokan}")
    print(f"Cocok           : {jumlah_cocok}")
    print(f"Berbeda         : {jumlah_berbeda}")
    print(f"Tidak ditemukan : {jumlah_tidak_ditemukan}")
    print(f"Akurasi         : {akurasi:.2f}%")
    print("=" * 50)

    kolom_tampil = [
        "sent_id",
        "token_id",
        "negasi_scope_patokan",
        "negasi_scope_cek"
    ]

    # Tambahkan token jika tersedia
    if "token_patokan" in berbeda.columns:
        kolom_tampil.insert(2, "token_patokan")
    elif "token" in berbeda.columns:
        kolom_tampil.insert(2, "token")

    print("\nDetail data berbeda:")
    if jumlah_berbeda > 0:
        display(berbeda[kolom_tampil])
    else:
        print("Tidak ada data berbeda.")

    print("\nDetail data tidak ditemukan:")
    if jumlah_tidak_ditemukan > 0:
        display(tidak_ditemukan[kolom_tampil])
    else:
        print("Tidak ada data yang tidak ditemukan.")

    return {
        "merged": merged,
        "cocok": cocok,
        "berbeda": berbeda,
        "tidak_ditemukan": tidak_ditemukan,
        "akurasi": akurasi
    }

In [17]:
filename_patokan, df_patokan = read_uploaded_file(upload_patokan)
filename_cek, df_cek = read_uploaded_file(upload_cek)

print(f"File patokan : {filename_patokan}")
print(f"File cek     : {filename_cek}")

hasil = cek_negasi_scope(df_patokan, df_cek)

File patokan : negasiScope_sample_train.xlsx
File cek     : negasiScope_sample_train_annotated.xlsx
HASIL PENGECEKAN NEGATION SCOPE DETECTION
Baris patokan   : 5414
Cocok           : 5403
Berbeda         : 11
Tidak ditemukan : 0
Akurasi         : 99.80%

Detail data berbeda:


,sent_id,token_id,token_patokan,negasi_scope_patokan,negasi_scope_cek
3495,train-s3537,12,ketidakcukupan,Out_Scope,Kata Negasi
3496,train-s3537,13,nuansa,Out_Scope,In_Scope
4019,train-s3853,14,bukannya,Out_Scope,Kata Negasi
4020,train-s3853,15,menciptakan,Out_Scope,In_Scope
4021,train-s3853,16,kerusakan,Out_Scope,In_Scope
4022,train-s3853,17,di,Out_Scope,In_Scope
4023,train-s3853,18,muka,Out_Scope,In_Scope
4024,train-s3853,19,bumi,Out_Scope,In_Scope
4291,train-s4048,2,ketidakpastian,Out_Scope,Kata Negasi
4292,train-s4048,3,tentang,Out_Scope,In_Scope



Detail data tidak ditemukan:
Tidak ada data yang tidak ditemukan.
